In [0]:
import json
import time
import pyspark.sql.functions as sf
from functools import reduce

CHECKPOINT_PATH = "/Volumes/dev/raw/checkpoints/ttc_trip_updates_raw_batch"
DATA_PATH = "/Volumes/dev/raw/ttc_trip_updates_volume_completed"

In [0]:
RAW_TABLE = "dev.raw.ttc_trip_updates_raw_batch"
QUERY_NAME = "ttc_trip_updates_batch_autoloader"

SCHEMA_LOCATION = f"{CHECKPOINT_PATH}/schema"
STREAM_CHECKPOINT_LOCATION = f"{CHECKPOINT_PATH}/stream"

In [0]:
def run_batch_ingest_once(
    source_path: str = DATA_PATH,
    target_table: str = RAW_TABLE,
    schema_location: str = SCHEMA_LOCATION,
    checkpoint_location: str = STREAM_CHECKPOINT_LOCATION,
    query_name: str = QUERY_NAME,
    include_existing_files: bool = True,
):
    existing_query = next(
        (stream for stream in spark.streams.active if stream.name == query_name),
        None,
    )

    if existing_query is not None:
        print(f"Stream '{query_name}' is already running.")
        return existing_query

    source_df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.includeExistingFiles", str(include_existing_files).lower())
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .load(source_path)
    )

    raw_df = (
        source_df.withColumn("_source_file", sf.input_file_name())
        .withColumn("_ingest_ts", sf.current_timestamp())
        .withColumn("_ingest_date", sf.to_date(sf.current_timestamp()))
    )

    query = (
        raw_df.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_location)
        .option("mergeSchema", "true")
        .queryName(query_name)
        .trigger(processingTime="1 hour")
        .toTable(target_table)
    )

    print(f"Started '{query_name}' and writing to {target_table}.")
    print("This run will process all currently available new files every hour.")
    print(f"Source: {source_path}")
    print(f"Schema tracking: {schema_location}")
    print(f"Checkpoint: {checkpoint_location}")
    return query

In [0]:
batch_query = run_batch_ingest_once()